# Natya Posture Alignment - Colab Training Runbook

Run this notebook in Google Colab to train the model using your custom dataset hosted in Google Drive.


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install required dependencies
!pip install mediapipe opencv-python-headless pandas scikit-learn seaborn matplotlib tqdm


In [ ]:
# 2. Imports and Configuration
import os, glob, pickle, warnings, re
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict
warnings.filterwarnings('ignore')

# ----------------- CONFIGURATION -----------------
# UPDATE THESE PATHS to match where you saved your data in Drive!
DRIVE_ROOT = '/content/drive/MyDrive/TrainingData'

VIDEOS_DIR = f'{DRIVE_ROOT}/FinalModelTrainingVidoes'
CSV_PATH = f'{DRIVE_ROOT}/Instructions/dynamic_steps_template.csv'
CHECKPOINT_DIR = f'{DRIVE_ROOT}/Checkpoints'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_FRAMES = 120
MIN_VISIBILITY = 0.50
MAX_BAD_JOINT_FRAC = 0.20
MIN_VIDEOS = 2
MAX_PER_CLASS = None

FEATURES_CACHE = f'{CHECKPOINT_DIR}/adavu_features.npz'
RAW_CACHE = f'{CHECKPOINT_DIR}/raw_samples.pkl'
CKPT_PATH = f'{CHECKPOINT_DIR}/dance_coach_model.pt'

print(f'Device: {DEVICE}')
print(f'Reading videos from: {VIDEOS_DIR}')
print(f'Reading CSV from: {CSV_PATH}')


In [ ]:
# 3. Angle Definitions and MediaPipe Utilities
ANGLE_DEFS = [
    ('left_shoulder',  13, 11, 23),
    ('right_shoulder', 14, 12, 24),
    ('left_elbow',     11, 13, 15),
    ('right_elbow',    12, 14, 16),
    ('left_wrist',     13, 15, 19),
    ('right_wrist',    14, 16, 20),
    ('left_hip',       11, 23, 25),
    ('right_hip',      12, 24, 26),
    ('left_knee',      23, 25, 27),
    ('right_knee',     24, 26, 28),
    ('left_ankle',     25, 27, 31),
    ('right_ankle',    26, 28, 32),
]
ANGLE_NAMES = [d[0] for d in ANGLE_DEFS]
NUM_ANGLES  = len(ANGLE_DEFS)

FEATURE_DIM = 132 + 36 + 6   # = 174

# MediaPipe Initialization
!wget -q -O pose_landmarker_heavy.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

base_options = python.BaseOptions(model_asset_path='pose_landmarker_heavy.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False,
    num_poses=1,
)
mp_pose = vision.PoseLandmarker.create_from_options(options)
print('MediaPipe model ready.')

def _angle_between(pa, pv, pc):
    v1 = pa - pv;  v2 = pc - pv
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return 0.0
    cos_a = np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_a)))

def compute_angles_for_frame(frame):
    return np.array([_angle_between(frame[a, :2], frame[v, :2], frame[c, :2]) for _, a, v, c in ANGLE_DEFS])

def normalise_landmarks(seq):
    seq = seq.copy()
    hip_mid      = (seq[:, 23, :2] + seq[:, 24, :2]) / 2
    shoulder_mid = (seq[:, 11, :2] + seq[:, 12, :2]) / 2
    scale        = np.linalg.norm(shoulder_mid - hip_mid, axis=1)
    scale        = np.maximum(scale, 1e-6)[:, np.newaxis]
    seq[:, :, :2] = (seq[:, :, :2] - hip_mid[:, np.newaxis, :]) / scale[:, np.newaxis, :]
    return seq

def pad_to_square(image: np.ndarray) -> np.ndarray:
    h, w = image.shape[:2]
    if h == w: return image
    size = max(h, w)
    pad_h = (size - h) // 2
    pad_w = (size - w) // 2
    return cv2.copyMakeBorder(image, pad_h, size - h - pad_h, pad_w, size - w - pad_w, cv2.BORDER_CONSTANT, value=[0, 0, 0])

def extract_landmarks_from_video(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return None, None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total < 2:
        cap.release()
        return None, None

    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    target_indices = set(indices)
    max_idx = max(target_indices) if target_indices else -1
    
    raw_seq_dict = {}
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx > max_idx: break
        if frame_idx in target_indices:
            frame = pad_to_square(frame)
            rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result   = mp_pose.detect(mp_image)
            if result.pose_landmarks:
                lm = result.pose_landmarks[0]
                arr = np.array([[l.x, l.y, l.visibility] for l in lm])
                bad_frac = np.mean(arr[:, 2] < MIN_VISIBILITY)
                if bad_frac <= MAX_BAD_JOINT_FRAC:
                    raw_seq_dict[frame_idx] = arr
        frame_idx += 1
    cap.release()
    raw_seq = [raw_seq_dict[idx] for idx in indices if idx in raw_seq_dict]
    if len(raw_seq) < 5: return None, None
    raw_seq   = np.array(raw_seq)
    seq_norm  = normalise_landmarks(raw_seq)
    angles_seq = np.array([compute_angles_for_frame(seq_norm[t]) for t in range(len(seq_norm))])
    return seq_norm, angles_seq

SYMMETRY_PAIRS = [
    (ANGLE_NAMES.index('left_shoulder'), ANGLE_NAMES.index('right_shoulder')),
    (ANGLE_NAMES.index('left_elbow'),    ANGLE_NAMES.index('right_elbow')),
    (ANGLE_NAMES.index('left_wrist'),    ANGLE_NAMES.index('right_wrist')),
    (ANGLE_NAMES.index('left_hip'),      ANGLE_NAMES.index('right_hip')),
    (ANGLE_NAMES.index('left_knee'),     ANGLE_NAMES.index('right_knee')),
    (ANGLE_NAMES.index('left_ankle'),    ANGLE_NAMES.index('right_ankle')),
]

def compute_symmetry_features(angles_mean):
    return np.array([abs(angles_mean[l] - angles_mean[r]) for l, r in SYMMETRY_PAIRS])

def build_feature_vector(seq_norm, angles_seq):
    coords = seq_norm[:, :, :2]
    coord_mean = coords.mean(axis=0).flatten()
    coord_std  = coords.std(axis=0).flatten()
    angle_mean = angles_seq.mean(axis=0)
    angle_std  = angles_seq.std(axis=0)
    angle_vel  = np.abs(np.diff(angles_seq, axis=0)).mean(axis=0)
    sym = compute_symmetry_features(angle_mean)
    return np.concatenate([coord_mean, coord_std, angle_mean, angle_std, angle_vel, sym])

# Augmentation Utils
LR_PAIRS = [(11, 12), (13, 14), (15, 16), (17, 18), (19, 20), (21, 22), (23, 24), (25, 26), (27, 28), (29, 30), (31, 32), (1, 4), (2, 5), (3, 6), (7, 8), (9, 10)]

def flip_sequence(seq_norm):
    flipped = seq_norm.copy()
    flipped[:, :, 0] *= -1
    for l, r in LR_PAIRS:
        flipped[:, [l, r]] = flipped[:, [r, l]]
    return flipped

def add_noise(seq_norm, sigma=0.01):
    noisy = seq_norm.copy()
    noisy[:, :, :2] += np.random.normal(0, sigma, noisy[:, :, :2].shape)
    return noisy

def speed_warp(seq_norm, angles_seq, factor=0.8):
    T = len(seq_norm)
    new_T   = max(5, int(T * factor))
    old_idx = np.linspace(0, T - 1, new_T)
    new_idx = np.linspace(0, new_T - 1, T)
    def interp_seq(s):
        out = np.zeros_like(s)
        for j in range(s.shape[1]):
            for k in range(s.shape[2]):
                sampled = np.interp(old_idx, np.arange(T), s[:, j, k])
                out[:, j, k] = np.interp(new_idx, np.arange(new_T), sampled)
        return out
    def interp_ang(a):
        out = np.zeros_like(a)
        for j in range(a.shape[1]):
            sampled = np.interp(old_idx, np.arange(T), a[:, j])
            out[:, j] = np.interp(new_idx, np.arange(new_T), sampled)
        return out
    return interp_seq(seq_norm), interp_ang(angles_seq)

def augment_sample(seq_norm, angles_seq):
    variants = []
    flipped = flip_sequence(seq_norm)
    flip_angles = np.array([compute_angles_for_frame(flipped[t]) for t in range(len(flipped))])
    variants.append((build_feature_vector(flipped, flip_angles), flip_angles.mean(axis=0), flip_angles.std(axis=0)))

    noisy = add_noise(seq_norm)
    noisy_angles = np.array([compute_angles_for_frame(noisy[t]) for t in range(len(noisy))])
    variants.append((build_feature_vector(noisy, noisy_angles), noisy_angles.mean(axis=0), noisy_angles.std(axis=0)))

    sw_seq, sw_ang = speed_warp(seq_norm, angles_seq, factor=1.2)
    variants.append((build_feature_vector(sw_seq, sw_ang), sw_ang.mean(axis=0), sw_ang.std(axis=0)))

    sw_seq2, sw_ang2 = speed_warp(seq_norm, angles_seq, factor=0.8)
    variants.append((build_feature_vector(sw_seq2, sw_ang2), sw_ang2.mean(axis=0), sw_ang2.std(axis=0)))

    return variants



In [ ]:
# 4. Read Labels and Map to Videos in Google Drive
print(f"Loading instructions from {CSV_PATH}")
df = pd.read_csv(CSV_PATH)
df['Step_Name'] = df['Step_Name'].astype(str).str.strip()
df['Step_ID'] = df['Step_ID'].astype(str).str.strip()

print(f"Loaded {len(df)} rows from CSV.")

# Find matching video file for each Step_ID in Drive
all_drive_videos = glob.glob(f"{VIDEOS_DIR}/*")
print(f"Found {len(all_drive_videos)} total items in the video directory.")

video_files = {}
for _, row in df.iterrows():
    step_id = row['Step_ID']
    label = row['Step_Name']
    
    # Simple search: if step_id is part of the filename
    matched_file = None
    for vf in all_drive_videos:
        if step_id in os.path.basename(vf):
            matched_file = vf
            break
            
    if matched_file:
        video_files[step_id] = {'path': matched_file, 'label': label}

print(f"Successfully matched {len(video_files)} videos in Drive.")

# Count classes
class_counts = defaultdict(int)
for v in video_files.values():
    class_counts[v['label']] += 1

print('\nClass counts (raw):')
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f'  {cls:<40s} {cnt:3d}')

valid_classes = {cls for cls, cnt in class_counts.items() if cnt >= MIN_VIDEOS}

# Filtered list to process
processing_list = []
for k, v in video_files.items():
    if v['label'] in valid_classes:
        processing_list.append(v)
        
print(f"\nKept {len(valid_classes)} classes with >= {MIN_VIDEOS} videos")
print(f"Total videos to process: {len(processing_list)}")



In [ ]:
# 5. Extract Features and Cache Data
cache_valid = False
if os.path.exists(FEATURES_CACHE):
    try:
        data = np.load(FEATURES_CACHE, allow_pickle=True)
        if data['X'].shape[1] == FEATURE_DIM:
            print(f'Cache found → {FEATURES_CACHE}')
            X = data['X']; y = data['y']
            label_names = list(data['label_names'])
            angle_means = data['angle_means']; angle_stds = data['angle_stds']
            print(f'Loaded {len(X)} samples, {len(label_names)} classes.')
            cache_valid = True
        else:
            print(f'Cache dim mismatch. Rebuilding...')
    except Exception as e:
        print(f'Failed to load cache: {e}')

raw_samples = []
if os.path.exists(RAW_CACHE):
    print(f'Loading raw sequences from {RAW_CACHE}...')
    try:
        with open(RAW_CACHE, 'rb') as f:
            raw_samples = pickle.load(f)
    except Exception as e:
        print(f'Failed to load raw cache: {e}')
        raw_samples = []

if not cache_valid:
    failed = []
    
    if len(raw_samples) == 0:
        print('Extracting raw video sequences (this might take a while)...')
        for item in tqdm(processing_list):
            vid_path = item['path']
            cls = item['label']
            try:
                seq_norm, angles_seq = extract_landmarks_from_video(vid_path)
                if seq_norm is None:
                    failed.append(vid_path)
                    continue
                fv = build_feature_vector(seq_norm, angles_seq)
                a_mu = angles_seq.mean(axis=0)
                a_sig = angles_seq.std(axis=0)
                raw_samples.append((fv, a_mu, a_sig, cls, seq_norm, angles_seq, vid_path))
            except Exception as e:
                failed.append(vid_path)
                print(f'FAILED: {vid_path} — {e}')

        print(f'Saving raw cache to {RAW_CACHE}...')
        with open(RAW_CACHE, 'wb') as f:
            pickle.dump(raw_samples, f)
            
    print(f'Raw samples available for augmentation: {len(raw_samples)}')
    
    # Extract labels to find max count for balancing
    temp_y = [item[3] for item in raw_samples]
    unique_labels = sorted(set(temp_y))
    class_sample_counts = {cls: int(np.sum(np.array(temp_y) == cls)) for cls in unique_labels}
    max_count = max(class_sample_counts.values()) if class_sample_counts else 0
    
    print(f'Balancing classes to max_count: {max_count}')
    
    # Augment
    X_aug, y_aug, am_aug, as_aug = [], [], [], []
    class_raw = defaultdict(list)
    for item in raw_samples:
        class_raw[item[3]].append(item)

    for cls in unique_labels:
        items = class_raw[cls]
        for fv, a_mu, a_sig, _, seq_norm, angles_seq, hf_path in items:
            X_aug.append(fv); y_aug.append(cls)
            am_aug.append(a_mu); as_aug.append(a_sig)

        needed = max_count - len(items)
        if needed <= 0: continue

        pool = items.copy()
        added = 0
        while added < needed:
            src = pool[added % len(pool)]
            fv, a_mu, a_sig, _, seq_norm, angles_seq, hf_path = src
            variants = augment_sample(seq_norm, angles_seq)
            for v_fv, v_amu, v_asig in variants:
                if added >= needed: break
                X_aug.append(v_fv); y_aug.append(cls)
                am_aug.append(v_amu); as_aug.append(v_asig)
                added += 1

    X = np.array(X_aug); y = np.array(y_aug)
    label_names = sorted(set(y))
    angle_means = np.array(am_aug); angle_stds = np.array(as_aug)

    np.savez(FEATURES_CACHE, X=X, y=y, label_names=label_names, angle_means=angle_means, angle_stds=angle_stds)
    print(f'\nSaved cache → {FEATURES_CACHE}')
    print(f'Samples: {len(X)} | Classes: {len(label_names)} | Failed: {len(failed)}')

# Build Reference Distributions
REGIONS = {
    'legs':  ['left_knee', 'right_knee', 'left_hip', 'right_hip', 'left_ankle', 'right_ankle'],
    'arms':  ['left_elbow', 'right_elbow', 'left_shoulder', 'right_shoulder', 'left_wrist', 'right_wrist'],
}
angle_refs = {}
if 'raw_samples' in locals() and raw_samples:
    class_raw = defaultdict(list)
    for item in raw_samples: class_raw[item[3]].append(item)
    for cls in label_names:
        items = class_raw.get(cls, [])
        if not items: continue
        orig_mus = np.array([item[1] for item in items])
        angle_refs[cls] = {
            'mean': orig_mus.mean(axis=0),
            'std':  np.maximum(orig_mus.std(axis=0), 3.0),
            'master_angles': items[0][5],
            'master_landmarks': items[0][4],
            'master_hf_path': items[0][6]
        }



In [ ]:
# 6. Model Definition and Training Loop
le = LabelEncoder()
y_enc = le.fit_transform(y)
NUM_CLASSES = len(le.classes_)

X_mean = X.mean(axis=0)
X_std  = X.std(axis=0) + 1e-8
X_norm = (X - X_mean) / X_std

try:
    X_train, X_val, y_train, y_val = train_test_split(X_norm, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
except ValueError:
    X_train, X_val, y_train, y_val = train_test_split(X_norm, y_enc, test_size=0.2, random_state=42)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Classes: {NUM_CLASSES}')

class AdavuClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, hidden=256, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, hidden // 4), nn.BatchNorm1d(hidden // 4), nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(hidden // 4, num_classes),
        )
    def forward(self, x): return self.net(x)

class AdavuDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

model = AdavuClassifier(FEATURE_DIM, NUM_CLASSES).to(DEVICE)

EPOCHS, BATCH, LR, WD = 200, 32, 3e-4, 1e-4

train_loader = DataLoader(AdavuDataset(X_train, y_train), batch_size=BATCH, shuffle=True, drop_last=True if len(X_train) > BATCH else False)
val_loader   = DataLoader(AdavuDataset(X_val, y_val), batch_size=BATCH, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

counts_tr = np.array([np.sum(y_train == i) for i in range(NUM_CLASSES)], dtype=float)
cw = 1.0 / (counts_tr + 1e-8)
cw = cw / cw.sum() * NUM_CLASSES
criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(cw).to(DEVICE))

train_losses, val_losses, val_accs = [], [], []
best_val_acc, best_state = 0.0, None

for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running += loss.item() * len(Xb)
    if len(train_loader) > 0: train_losses.append(running / len(X_train))
    scheduler.step()

    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            val_loss += criterion(logits, yb).item() * len(Xb)
            correct  += (logits.argmax(1) == yb).sum().item()
    if len(val_loader) > 0: val_losses.append(val_loss / len(X_val))
    acc = correct / len(X_val) if len(X_val) > 0 else 0
    val_accs.append(acc)

    if acc > best_val_acc:
        best_val_acc = acc
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 25 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | val_acc={acc:.2%}')

if best_state: model.load_state_dict(best_state)
print(f'\nBest val accuracy: {best_val_acc:.2%}')



In [ ]:
# 7. Evaluation and Save Checkpoint
if best_state is None:
    print("Training failed or did not run.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    if train_losses: axes[0].plot(train_losses, label='Train loss')
    if val_losses: axes[0].plot(val_losses, label='Val loss')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
    axes[1].plot(val_accs)
    axes[1].axhline(best_val_acc, color='r', linestyle='--', label=f'Best {best_val_acc:.2%}')
    axes[1].set_title('Val Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(f'{CHECKPOINT_DIR}/training_curves.png', dpi=150)
    
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            preds = model(Xb.to(DEVICE)).argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(yb.numpy())

    print(classification_report(all_true, all_preds, target_names=le.classes_))
    
    cm = confusion_matrix(all_true, all_preds)
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, ax=ax, cmap='Blues')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(f'{CHECKPOINT_DIR}/confusion_matrix.png', dpi=150)
    
    ckpt = {
        'model_state':  best_state,
        'label_encoder': le,
        'X_mean':       X_mean,
        'X_std':        X_std,
        'num_classes':  NUM_CLASSES,
        'feature_dim':  FEATURE_DIM,
        'angle_refs':   angle_refs if 'angle_refs' in locals() else {},
        'angle_names':  ANGLE_NAMES,
        'angle_defs':   ANGLE_DEFS,
        'regions':      REGIONS if 'REGIONS' in locals() else {},
        'best_val_acc': best_val_acc,
    }
    torch.save(ckpt, CKPT_PATH)
    print(f'Checkpoint saved -> {CKPT_PATH}')

